# Corpus-regression MaxRL

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.corpus_regression.maxrl import CorpusRegressionMaxRLConfig
from src.experiments.corpus_regression.utils import dim_averaged_metrics_from_parquet

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/corpus_regression/maxrl.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


### Configs

In [ ]:
config = CorpusRegressionMaxRLConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "corpus-regression",
    study_base_folder=repo_root / "artifacts" / "corpus-regression-maxrl-example",
    num_lookforward_tokens=4,
    train_epochs=2,
    num_rollouts_per_sample=128,
    gaussian_stdev=1.0,
    subtract_baseline=True,
    use_factorized_likelihoods=True,
)

display(config.visualize())

In [ ]:
state = config.initialize(device=device)
state.run_training()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

maxrl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

maxrl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

### Results

In [ ]:
metrics = pl.read_parquet(config.study_folder / "metrics.parquet")
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,val_target_xx,val_target_xy,val_target_yy,val_target_n
i64,list[f64],list[f64],list[f64],f64,list[f64],list[f64],list[f64],f64
0,"[17360.849609, 15141.858398, … 14343.59375]","[1450.759033, 799.55011, … 813.387634]","[49984.0, 49984.0, … 49984.0]",49984.0,"[5638.060059, 1221.981689, … 14033.006836]","[1674.973511, 256.794891, … 2062.69458]","[49920.0, 49920.0, … 49920.0]",49920.0
1,"[8750.495117, 6000.012207, … 7990.630859]","[1442.134521, 1484.052734, … 1359.336792]","[49984.0, 49984.0, … 49984.0]",49984.0,"[8432.349609, 8554.254883, … 12207.217773]","[2340.762695, 2739.513428, … 2682.226074]","[49920.0, 49920.0, … 49920.0]",49920.0


In [ ]:
dim_averaged_metrics_from_parquet(metrics, split="train").join(
    dim_averaged_metrics_from_parquet(metrics, split="val"), on="epoch"
)

epoch,train_avg_corr,train_avg_rsq,val_avg_corr,val_avg_rsq
i64,f64,f64,f64,f64
0,0.037879,-0.256158,0.063597,-0.11985
1,0.075399,-0.098636,0.089457,-0.083334


In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_df = pl.read_parquet(
    config.study_folder / str(last_epoch) / "validation.parquet"
)
validation_df.head()

model_preds,target
list[f64],list[f64]
"[-0.363281, 0.703125, … -0.707031]","[-1.0, 1.0, … -1.0]"
"[-0.335938, 0.498047, … -0.582031]","[1.0, -1.0, … 1.0]"
"[-0.337891, 0.355469, … -0.300781]","[1.0, 1.0, … 1.0]"
"[-0.251953, 0.285156, … -0.316406]","[1.0, -1.0, … -1.0]"
"[-0.349609, 0.451172, … -0.582031]","[-1.0, -1.0, … -1.0]"
